<a href="https://colab.research.google.com/github/SWATHIRAVIRS/fake_news_detection/blob/main/Copy_of_fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
!unzip data.csv.zip

In [ ]:
#cleaning
import pandas as pd
import torch
import evaluate
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
df=pd.read_csv("/content/data.csv.zip")

In [ ]:
df.head()

In [ ]:
type(df)

In [ ]:
df['text']=df["Headline"] + " " + df["Body"]

In [ ]:
df=df[["text","Label"]]

In [ ]:
df.head()

In [ ]:
df["Label"].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df=df.dropna()

In [ ]:
df.isnull().sum()

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42)

In [ ]:
from datasets import Dataset
train_dataset=Dataset.from_pandas(train_df.drop(columns=['__index_level_0__'],errors='ignore'))
test_dataset=Dataset.from_pandas(test_df.drop(columns=['__index_level_0__'],errors='ignore'))

In [ ]:
#tokenzier
model_checkpoint="distilbert-base-uncased"
tokenzier=AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
#tokenization
def preprocess_function(examples):
  result = tokenzier(
      examples["text"],
      truncation=True,
      padding=True,
      max_length=512
  )
  result["labels"]=examples["Label"]
  return result

In [ ]:
tokenized_train=train_dataset.map(preprocess_function,batched=True,remove_columns=train_dataset.column_names)
tokenized_test=test_dataset.map(preprocess_function,batched=True,remove_columns=test_dataset.column_names)

In [ ]:
# Rename 'Label' to 'labels' so the model can find it to calculate loss
tokenized_train = tokenized_train.set_format("torch",columns=["input_ids","attention_mask", "labels"])
tokenized_test = tokenized_test.set_format("torch",columns=["input_ids","attention_mask", "labels"])

In [ ]:
#model selection
from transformers import AutoModelForSequenceClassification
model=AutoModelForSequenceClassification.from_pretrained(model_checkpoint,num_labels=2)
data_collator = DataCollatorWithPadding(tokenizer=tokenzier)

In [ ]:
import numpy as np
metrices=evaluate.load("accuracy")
def compute_metrices(eval_pred):
  logits,label = eval_pred
  predictions=np.argmax(logits,axis=1)
  return metrices.compute(predictions=predictions,references=label)

In [ ]:
#training setup
from transformers import TrainingArguments
training_args=TrainingArguments(
    output_dir="/fake_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model is running on:{device}")

In [ ]:
def preprocess_function(examples):
    result = tokenzier(examples["text"], truncation=True, padding="max_length", max_length=512)
    result["labels"] = examples["Label"]
    return result

final_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
final_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

# 2. Force the Trainer to use these new variables
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train,  # Using the 'final_' versions we just made
    eval_dataset=final_test,    # This WILL NOT be None anymore
    data_collator=DataCollatorWithPadding(tokenizer=tokenzier),
    compute_metrics=compute_metrices,
)

print(f"Dataset ready! Training on {len(final_train)} samples...")
trainer.train()

In [ ]:
from transformers import pipeline
news_detector = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenzier,
    device=0 if torch.cuda.is_available() else -1
)

def check_news(text):
    result = news_detector(text)[0]
    label = result['label']
    score = result['score']
    status = "REAL" if label == "LABEL_1" else "FAKE"

    print(f"Prediction: {status}")
    print(f"Confidence: {score:.2%}")

my_headline = "NASA discovers life on Mars inside a chocolate bar."
check_news(my_headline)